<img src="https://userweb.fct.unl.pt/~jmc.xavier/MSII/iLogos/logo_novafct.png" width="200">

# Departamento de Engenharia Mecânica e Industrial
## Mecânica dos Sólidos II

## Deslocamentos transversais de vigas

### Problema 1

A viga AD apoia-se na viga EF como indicado. Sabendo que é utilizado um perfil HEB 300 para as construir, determine a flecha nos pontos B e C para o carregamento ilustrado. Considere $E = 200$ GPa.

<img src="https://userweb.fct.unl.pt/~jmc.xavier/MSII/Notebooks/Au08/P1/MSII_Au08_P1.png"
width="500">


In [1]:
import numpy as np
import sympy as sy
from sympy.solvers import solve
import matplotlib.pyplot as plt
import os

cor = '2'
if cor == '1':
    plt.rcParams['axes.facecolor'] = (.15, .15, .15)
    plt.rcParams['figure.facecolor'] = (.15, .15, .15)
    plt.rcParams['font.family'] = 'monospace'
    plt.rcParams['font.size'] = 18
    # plt.rcParams['text.usetex'] = True
    params = {"ytick.color" : (.8, .8, .8),
              "xtick.color" : (.8, .8, .8),
              "grid.color" : (.2, .2, .2),
              "text.color" : (.7, .7, .7),
              "axes.labelcolor" : (.8, .8, .8),
              "axes.edgecolor" : (.15, .15, .15)}
else:
    plt.rcParams['axes.facecolor'] = (.7, .7, .7)
    plt.rcParams['figure.facecolor'] = (.7, .7, .7)
    plt.rcParams['font.family'] = 'monospace'
    plt.rcParams['font.size'] = 18
    # plt.rcParams['text.usetex'] = True
    params = {"ytick.color" : (.1, .1, .1),
              "xtick.color" : (.1, .1, .1),
              "grid.color" : (.2, .2, .2),
              "text.color" : (.1, .1, .1),
              "axes.labelcolor" : (.1, .1, .1),
              "axes.edgecolor" : (.15, .15, .15)}
plt.rcParams.update(params)

# data structure, units: N, mm, MPa

class varin: pass

d = varin()
HEB300 = varin()

d.L = 1.  # unit: m
d.PB = 60.e3 # unit: N
d.PC = 60.e3 # unit: N

d.E = 200.e9 # unit: Pa
HEB300.Iz = 252.e-6 # unit: m⁴

### Resolução

#### Viga AD (Formulário Anexo 1)

Considere-se em primeiro lugar a viga AD assumindo apaios fixos. Pelo princípio da sobreposição pode-se analisar a deflexão na secção B e C pelos efeitos combinados da carga em B ($P_B$) e C ($P_C$):

\begin{equation*}
\delta_B = \delta_B (P_B) + \delta_B (P_C)
\quad\wedge\quad
\delta_C = \delta_C (P_B) + \delta_C (P_C)
\end{equation*}

Por consulta direta de tabelas pode-se determinar a deflexão em cada secção para cada cenário de carregamento.

#### $\delta_B (P_B) \equiv \delta_C (P_C)$

Por consulta a tabelas obtêm-se diretamente as expressões do deslocamento transverso,

\begin{equation*}
\textrm{para $x=a$} \quad\rightarrow\quad \delta_B (P_B) =
\frac{Pa^2b^2}{3LEI}
\quad\wedge\quad
a = L_{AB} = 1~\textrm{m}
\quad\wedge\quad
b = L_{BD} = 2~\textrm{m}
\end{equation*}

In [2]:
p, a, b, e, i, l = sy.symbols('p a b e i l')

def deltaP1(p,a,b,e,i,l):
    return p*a**2*b**2/3/e/i/l

deltaB_PB = deltaP1(d.PB,d.L,2*d.L,d.E,HEB300.Iz,3*d.L)
print('::: delta B - PB :::')
print(f'deltaB(PB) = {deltaB_PB:.3e} [m]')
print(':: por simetria do problema ::')
deltaC_PC = deltaB_PB #
print(f'deltaC(PC) = deltaB(PB) = {deltaC_PC:.3e} [m]')

::: delta B - PB :::
deltaB(PB) = 5.291e-04 [m]
:: por simetria do problema ::
deltaC(PC) = deltaB(PB) = 5.291e-04 [m]


#### $\delta_C (P_B) \equiv \delta_B (P_C)$

Neste caso é necessário usar um formulário considerando a origem em D,

\begin{equation*}
\delta_C (P_B) =
\frac{Pbx}{6LEI}(L² - b² - x)\Big|_{x=x_{DC}}
\quad\wedge\quad
b = L_{AB} = 1~\textrm{m}
\quad\wedge\quad
x = L_{DC} = 1~\textrm{m}
\end{equation*}

In [3]:
x = sy.symbols('x')

def deltaP2(p,b,x,e,i,l):
    return p*b*x/6/e/i/l*(l**2 - b**2 - x**2)

deltaC_PB = deltaP2(d.PB,d.L,d.L,d.E,HEB300.Iz,3*d.L)
print('::: delta C - PB :::')
print(f'deltaC(PB) = {deltaC_PB:.3e} [m]')
print(':: por simetria do problema ::')
deltaB_PC = deltaC_PB
print(f'deltaB(PC) = deltaC(PB) = {deltaB_PC:.3e} [m]')

::: delta C - PB :::
deltaC(PB) = 4.630e-04 [m]
:: por simetria do problema ::
deltaB(PC) = deltaC(PB) = 4.630e-04 [m]


#### Viga AD (Formulário Anexo 2)

<img src="https://userweb.fct.unl.pt/~jmc.xavier/MSII/Notebooks/Au08/P1/Anexo2simples.png"
width="700">

In [4]:
def deltaP1_v2(p,a,b,x,e,i,l):
    return p*b/6/e/i/l*( (a**2 + 2*a*b)*x - x**3)

deltaB_PB_2 = deltaP1_v2(d.PB,d.L,2*d.L,d.L,d.E,HEB300.Iz,3*d.L)
print('::: delta B - PB :::')
print(f'deltaB(PB) [formula 2] = {deltaB_PB_2:.3e} [m]')

::: delta B - PB :::
deltaB(PB) [formula 2] = 5.291e-04 [m]


In [5]:
def deltaP2_v2(p,a,x,e,i,l):
    return p*a/6/e/i/l*(-l*a**2 + (a**2 + 2*l**2)*x - 3*l*x**2 + x**3)

deltaC_PB_2 = deltaP2_v2(d.PB,d.L,2*d.L,d.E,HEB300.Iz,3*d.L)
print('::: delta C - PB :::')
print(f'deltaC(PB) [formula 2] = {deltaC_PB_2:.3e} [m]')

::: delta C - PB :::
deltaC(PB) [formula 2] = 4.630e-04 [m]


#### Viga EF


<img src="https://userweb.fct.unl.pt/~jmc.xavier/MSII/Notebooks/Au08/P1/MSII_Au08_P1_Deforrmed.png"
width="500">

Os deslocamentos acima determinados para a secção B e C consideram que o apoio em A tem deslocamento vertical nulo (apoio). Contudo, o próprio apoio A irá deslocar-se devido à deflexão da viga EF. Este deslocamento será sentido pela viga AD como um assentamento do apoio em A. A Viga irá rodar, em movimento de corpo rígido em torno do ponto D. Existirá por isso uma variação linear do deslocamento vertical entre a secção A e D.

No Diagrama de corpo livre da viga EF, sobre os pontos A e D surgem forças aplicadas de igual intensidade e sentido oposto em relação aos apoios da viga AD.

Determine-se a deflexão da viga EF no ponto A, devido ao carregamento vertical na secção A ($R_A$).

Por consulta a tabelas obtêm-se diretamente as expressões do deslocamento transverso,

\begin{equation*}
\delta_A^{R_A} =
\frac{Pa^2b^2}{3LEI}
~\wedge~
P \equiv R_A = 60~\textrm{kN}
~\wedge~
a = L_{EA} = 1~\textrm{m}
~\wedge~
b = L_{AF} = 3~\textrm{m}
\end{equation*}

In [6]:
deltaA_PB = deltaP1(d.PB,d.L,3*d.L,d.E,HEB300.Iz,4*d.L)
print('::: delta A - PB :::')
print(f'deltaA(PB) = {deltaA_PB:.3e} [m]')

::: delta A - PB :::
deltaA(PB) = 8.929e-04 [m]


O assentamento do apoio em A, devido à deflexão da viga EF (onde a viga AD se apoia), irá introduzir um deslocamento que varia linearmente entre a secção A (máximo) e a secção D (nulo).

Pela relação geométrica de igualdade de triângulos resulta,

\begin{equation*}
\frac{\delta_A^{R_A}}{L_{AD}} =
\frac{\delta_B^{\delta_A^{R_A}}}{  L_{BD}} =
\frac{\delta_C^{\delta_A^{R_A}}}{  L_{CD}}
\end{equation*}

de onde resulta,

\begin{equation*}
\delta_B^{\delta_A^{R_A}} =
\frac{L_{BD}}{L_{AD}} \delta_A^{R_A} =
\frac{2}{3} \delta_A^{R_A}
\end{equation*}

In [7]:
deltaB_deltaA = 2/3*deltaA_PB
print('::: delta B (delta A) :::')
print(f'deltaB(deltaA) = {deltaB_deltaA:.3e} [m]')

::: delta B (delta A) :::
deltaB(deltaA) = 5.952e-04 [m]


\begin{equation*}
\delta_C^{\delta_A^{R_A}} =
\frac{L_{CD}}{L_{AD}} \delta_A^{R_A} =
\frac{1}{3} \delta_A^{R_A}
\end{equation*}

In [8]:
deltaC_deltaA = 1/3*deltaA_PB
print('::: delta C (delta A) :::')
print(f'deltaC(deltaA) = {deltaC_deltaA:.3e} [m]')

::: delta C (delta A) :::
deltaC(deltaA) = 2.976e-04 [m]


#### Secção B: deslocamento total

- Princípio da sobreposição

Pelo princípio da sobreposição pela aplicação das 2 cargas de igual valor em B e C, resulta:

\begin{equation*}
P_B+P_C+\delta_A^{R_A}
\quad\longrightarrow\quad
\delta_B =
\delta_B (P_B) + \delta_B (P_C) +
\delta_B^{\delta_A^{R_A}}
\end{equation*}

In [9]:
print(':: pelo principio da sobreposicao ::')
dalta_B_total = deltaB_PB + deltaB_PC + deltaB_deltaA
print(f'dalta_B_total = deltaB_PB + deltaB_PC + deltaB_deltaA')
print(f'    = {deltaB_PB:.3e} + {deltaB_PC:.3e} + {deltaB_deltaA:.3e} = {dalta_B_total:.3e} [m]')

:: pelo principio da sobreposicao ::
dalta_B_total = deltaB_PB + deltaB_PC + deltaB_deltaA
    = 5.291e-04 + 4.630e-04 + 5.952e-04 = 1.587e-03 [m]


#### Secção C: deslocamento total

- Princípio da sobreposição

Pelo princípio da sobreposição pela aplicação das 2 cargas de igual valor em B e C, resulta:

\begin{equation*}
P_B+P_C+\delta_A^{R_A}
\quad\longrightarrow\quad
\delta_C =
\delta_C (P_B) + \delta_C (P_C) +
\delta_C^{\delta_A^{R_A}}
\end{equation*}

In [10]:
print(':: pelo principio da sobreposicao ::')
dalta_C_total = deltaB_PB + deltaB_PC + deltaB_deltaA
print(f'dalta_C_total = deltaC_PB + deltaC_PC + deltaC_deltaA')
print(f'    = {deltaC_PB:.3e} + {deltaC_PC:.3e} + {deltaC_deltaA:.3e} = {dalta_C_total:.3e} [m]')

:: pelo principio da sobreposicao ::
dalta_C_total = deltaC_PB + deltaC_PC + deltaC_deltaA
    = 4.630e-04 + 5.291e-04 + 2.976e-04 = 1.587e-03 [m]


#### Anexo 1

<img src="https://userweb.fct.unl.pt/~jmc.xavier/MSII/Notebooks/Formulario/BeamDeflectionSlopes.png"
width="900">

#### Anexo 2

<img src="https://userweb.fct.unl.pt/~jmc.xavier/MSII/Notebooks/Formulario/Form_Deflexao_CargaF.png"
width="600">

---

Copyright (c) [Mecânica dos Sólidos I - DEMI - NOVA-SST]

Interactive computing by <a href="https://jupyter.org/" target="_blank"> <span
style="color:#333399"> Jupyter Notebook </span> </a> &nbsp;|&nbsp;Coded by <a href = "mailto: jmc.xavier@fct.unl.pt">José Xavier</a>

Licensed under  <a href="http://creativecommons.org/licenses/by-sa/4.0/"
target="_blank"> <span style="color:#333399;font-size: 20px"> CC BY-SA 4.0  </span></a>
